In [1]:
import sys
sys.path.append("..")

from src.data_loader import load_clean_data, get_features_and_target

df_clean = load_clean_data()
X, y = get_features_and_target(df_clean)

In [2]:
X.shape, y.shape


((5758, 8), (5758,))

In [4]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [5]:
# Define all the models
models = {
    'kNN': Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=24))]),
    'Logistic Regression': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression())]),
    'Random Forest': Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier(random_state=42))]),
    'XGBoost': Pipeline([('scaler', StandardScaler()), ('clf', XGBClassifier(random_state=42))]),
}

In [6]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scoring = ['accuracy', 'f1', 'roc_auc']

results = {}
for name, pipe in models.items():
    scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring)
    results[name] = scores

In [9]:
results.items()

dict_items([('kNN', {'fit_time': array([0.00679517, 0.00284863, 0.        , 0.00207114, 0.01466966,
       0.01199579, 0.01732326, 0.0166986 , 0.0167737 , 0.01545358]), 'score_time': array([0.04611611, 0.04559565, 0.04713488, 0.05011773, 0.04404402,
       0.03023219, 0.03123593, 0.03327632, 0.03336859, 0.03687191]), 'test_accuracy': array([0.68229167, 0.67881944, 0.75520833, 0.71006944, 0.68923611,
       0.73784722, 0.71180556, 0.71354167, 0.67826087, 0.7426087 ]), 'test_f1': array([0.72481203, 0.72592593, 0.78603945, 0.74268105, 0.73481481,
       0.76946565, 0.74303406, 0.7480916 , 0.72180451, 0.77844311]), 'test_roc_auc': array([0.7477107 , 0.75481459, 0.80931045, 0.7676494 , 0.75129635,
       0.77983451, 0.77160895, 0.77222801, 0.75193846, 0.79369559])}), ('Logistic Regression', {'fit_time': array([0.164294  , 0.02009606, 0.01639891, 0.01735973, 0.01650524,
       0.0137887 , 0.01703   , 0.01675534, 0.01910663, 0.02962589]), 'score_time': array([0.01694489, 0.0130806 , 0.0166361

In [8]:
# Compiling the results

summary = pd.DataFrame({
    name: {
        'accuracy_mean': res['test_accuracy'].mean(),
        'accuracy_std': res['test_accuracy'].std(),
        'f1_mean': res['test_f1'].mean(),
        'f1_std': res['test_f1'].std(),
        'roc_auc_mean': res['test_roc_auc'].mean(),
        'roc_auc_std': res['test_roc_auc'].std(),
    }
    for name, res in results.items()
}).T

summary

,accuracy_mean,accuracy_std,f1_mean,f1_std,roc_auc_mean,roc_auc_std
kNN,0.709969,0.026602,0.747511,0.021841,0.770009,0.019026
Logistic Regression,0.652308,0.021767,0.709821,0.020633,0.713037,0.020902
Random Forest,0.748177,0.010085,0.778905,0.009837,0.820273,0.015025
XGBoost,0.737928,0.013331,0.773531,0.012425,0.805089,0.016508
